<a href="https://www.kaggle.com/code/abidur14004/illinosis-doc?scriptVersionId=267761395" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from tqdm.auto import tqdm
import warnings
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import json
from datetime import datetime
from scipy import stats
warnings.filterwarnings('ignore')


# ==================== VIT IMAGE PREPROCESSING CONFIGURATION ====================
class ViTPreprocessingConfig:
    """Configuration for ViT preprocessing with multi-modal features"""
    def __init__(self):
        # Image settings
        self.image_size = 224
        self.channels = 3
        
        # ImageNet normalization
        self.mean = [0.485, 0.456, 0.406]
        self.std = [0.229, 0.224, 0.225]
        
        # Data augmentation settings
        self.horizontal_flip_prob = 0.5
        self.color_jitter = {
            'brightness': 0.2,
            'contrast': 0.2,
            'saturation': 0.1,
            'hue': 0.05
        }
        self.rotation_degrees = 10
        
        # Cross-validation
        self.n_folds = 3
        self.random_state = 42
        
        # BMI filtering
        self.bmi_range = (16, 45)
        
        # Image quality thresholds
        self.min_image_size = 1
        
        # Feature columns to use
        self.feature_columns = ['race', 'eyes', 'sex', 'hair']
        
        # Feature engineering settings
        self.reference_date = datetime(2025, 1, 1)  # Adjust based on data collection date
        self.age_bins = [0, 18, 25, 35, 45, 55, 65, 100]
        self.age_labels = ['<18', '18-24', '25-34', '35-44', '45-54', '55-64', '65+']
        self.bmi_bins = [0, 18.5, 25, 30, 35, 100]
        self.bmi_labels = ['Underweight', 'Normal', 'Overweight', 'Obese-I', 'Obese-II+']


# ==================== ADVANCED FEATURE ENGINEERING ====================

def calculate_age(birth_date, reference_date=None):
    """Calculate age from date of birth"""
    if pd.isna(birth_date):
        return np.nan
    
    if reference_date is None:
        reference_date = datetime.now()
    
    try:
        if isinstance(birth_date, str):
            birth_date = pd.to_datetime(birth_date, errors='coerce')
        
        if pd.isna(birth_date):
            return np.nan
            
        age = reference_date.year - birth_date.year
        
        # Adjust for birthday not yet occurred
        if reference_date.month < birth_date.month or \
           (reference_date.month == birth_date.month and reference_date.day < birth_date.day):
            age -= 1
            
        return age if age >= 0 else np.nan
    except:
        return np.nan


def calculate_bmi(height_inches, weight_lbs):
    """Calculate BMI from height (inches) and weight (pounds)"""
    if pd.isna(height_inches) or pd.isna(weight_lbs):
        return np.nan
    if height_inches <= 0 or weight_lbs <= 0:
        return np.nan
    
    height_m = height_inches * 0.0254
    weight_kg = weight_lbs * 0.453592
    bmi = weight_kg / (height_m ** 2)
    return bmi


def calculate_bsa(height_inches, weight_lbs):
    """Calculate Body Surface Area using DuBois formula"""
    if pd.isna(height_inches) or pd.isna(weight_lbs):
        return np.nan
    
    height_cm = height_inches * 2.54
    weight_kg = weight_lbs * 0.453592
    
    # DuBois formula
    bsa = 0.007184 * (height_cm ** 0.725) * (weight_kg ** 0.425)
    return bsa


def calculate_ponderal_index(height_inches, weight_lbs):
    """Calculate Ponderal Index (alternative to BMI)"""
    if pd.isna(height_inches) or pd.isna(weight_lbs):
        return np.nan
    if height_inches <= 0:
        return np.nan
    
    height_m = height_inches * 0.0254
    weight_kg = weight_lbs * 0.453592
    
    ponderal = weight_kg / (height_m ** 3)
    return ponderal


def engineer_body_composition_features(df):
    """Engineer advanced body composition features"""
    df = df.copy()
    
    # Basic BMI
    if 'height' in df.columns and 'weight' in df.columns:
        df['bmi'] = df.apply(lambda row: calculate_bmi(row['height'], row['weight']), axis=1)
        
        # Body Surface Area
        df['bsa'] = df.apply(lambda row: calculate_bsa(row['height'], row['weight']), axis=1)
        
        # Ponderal Index
        df['ponderal_index'] = df.apply(lambda row: calculate_ponderal_index(row['height'], row['weight']), axis=1)
        
        # Height in meters and weight in kg for easier calculations
        df['height_m'] = df['height'] * 0.0254
        df['weight_kg'] = df['weight'] * 0.453592
        
        # Body shape indicator (ratio of weight to height squared)
        df['weight_height_ratio'] = df['weight_kg'] / (df['height_m'] ** 2)
        
        # Relative height (normalized within dataset)
        df['height_zscore'] = stats.zscore(df['height'].fillna(df['height'].median()))
        df['weight_zscore'] = stats.zscore(df['weight'].fillna(df['weight'].median()))
        
        # Body type indicator
        df['body_type_score'] = df['weight_zscore'] - df['height_zscore']
        
    return df


def engineer_demographic_features(df, config):
    """Engineer demographic features"""
    df = df.copy()
    
    # Calculate age from date of birth
    if 'birth' in df.columns or 'dob' in df.columns or 'date_of_birth' in df.columns:
        birth_col = next((col for col in ['birth', 'dob', 'date_of_birth'] if col in df.columns), None)
        if birth_col:
            df['age'] = df[birth_col].apply(lambda x: calculate_age(x, config.reference_date))
            
            # Age groups
            df['age_group'] = pd.cut(df['age'], bins=config.age_bins, labels=config.age_labels, right=False)
            
            # Age squared (for polynomial features)
            df['age_squared'] = df['age'] ** 2
            
            # Age decade
            df['age_decade'] = (df['age'] // 10) * 10
            
            # Age-BMI interaction
            if 'bmi' in df.columns:
                df['age_bmi_interaction'] = df['age'] * df['bmi']
                df['age_weight_interaction'] = df['age'] * df['weight_kg']
    
    return df


def engineer_bmi_categories(df, config):
    """Engineer BMI category features"""
    df = df.copy()
    
    if 'bmi' in df.columns:
        # Standard BMI categories
        df['bmi_category'] = pd.cut(df['bmi'], 
                                    bins=config.bmi_bins, 
                                    labels=config.bmi_labels, 
                                    right=False)
        
        # Binary health risk indicators
        df['is_underweight'] = (df['bmi'] < 18.5).astype(int)
        df['is_normal_weight'] = ((df['bmi'] >= 18.5) & (df['bmi'] < 25)).astype(int)
        df['is_overweight'] = ((df['bmi'] >= 25) & (df['bmi'] < 30)).astype(int)
        df['is_obese'] = (df['bmi'] >= 30).astype(int)
        
        # Distance from ideal BMI (22.5 is considered ideal)
        df['bmi_distance_from_ideal'] = np.abs(df['bmi'] - 22.5)
        
        # BMI squared (for polynomial relationships)
        df['bmi_squared'] = df['bmi'] ** 2
        
        # Log BMI (for log-linear relationships)
        df['bmi_log'] = np.log(df['bmi'].clip(lower=1))
        
        # BMI percentile within dataset
        df['bmi_percentile'] = df['bmi'].rank(pct=True) * 100
        
    return df


def engineer_categorical_interactions(df):
    """Engineer interaction features between categorical variables"""
    df = df.copy()
    
    # Race-Sex interaction
    if 'race' in df.columns and 'sex' in df.columns:
        df['race_sex'] = df['race'].astype(str) + '_' + df['sex'].astype(str)
    
    # Race-Age interaction
    if 'race' in df.columns and 'age_group' in df.columns:
        df['race_age'] = df['race'].astype(str) + '_' + df['age_group'].astype(str)
    
    # Eye-Hair interaction
    if 'eyes' in df.columns and 'hair' in df.columns:
        df['eye_hair_combo'] = df['eyes'].astype(str) + '_' + df['hair'].astype(str)
    
    # Sex-BMI category interaction
    if 'sex' in df.columns and 'bmi_category' in df.columns:
        df['sex_bmi_category'] = df['sex'].astype(str) + '_' + df['bmi_category'].astype(str)
    
    return df


def engineer_health_risk_features(df):
    """Engineer health risk indicators"""
    df = df.copy()
    
    if 'bmi' in df.columns and 'age' in df.columns:
        # Age-adjusted BMI risk
        # Higher risk for obesity in younger people
        df['age_adjusted_bmi_risk'] = df['bmi'] * (1 + (50 - df['age'].clip(0, 50)) / 100)
        
        # Metabolic syndrome risk indicator (simplified)
        df['metabolic_risk_score'] = 0
        
        # Risk increases with BMI > 25
        df.loc[df['bmi'] >= 25, 'metabolic_risk_score'] += 1
        df.loc[df['bmi'] >= 30, 'metabolic_risk_score'] += 1
        df.loc[df['bmi'] >= 35, 'metabolic_risk_score'] += 1
        
        # Risk increases with age > 40
        df.loc[df['age'] >= 40, 'metabolic_risk_score'] += 1
        df.loc[df['age'] >= 50, 'metabolic_risk_score'] += 1
        
    return df


def engineer_image_path_features(df):
    """Engineer features from image paths"""
    df = df.copy()
    
    if 'image_path' in df.columns:
        # Extract image type from path (front, side, inmates)
        df['image_type'] = df['image_path'].apply(
            lambda x: 'front' if 'front' in str(x).lower() 
            else ('side' if 'side' in str(x).lower() 
            else ('inmates' if 'inmates' in str(x).lower() else 'unknown'))
        )
        
        # Binary indicators for image type
        df['has_front_image'] = (df['image_type'] == 'front').astype(int)
        df['has_side_image'] = (df['image_type'] == 'side').astype(int)
        
    return df


def engineer_statistical_features(df):
    """Engineer statistical features"""
    df = df.copy()
    
    numeric_cols = ['bmi', 'height', 'weight', 'age']
    existing_cols = [col for col in numeric_cols if col in df.columns]
    
    for col in existing_cols:
        if df[col].notna().sum() > 0:
            # Z-score (already done for some, but comprehensive)
            df[f'{col}_zscore'] = stats.zscore(df[col].fillna(df[col].median()))
            
            # Percentile rank
            df[f'{col}_percentile'] = df[col].rank(pct=True) * 100
            
            # Quartile assignment
            df[f'{col}_quartile'] = pd.qcut(df[col], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')
            
            # Deviation from median
            df[f'{col}_deviation_from_median'] = df[col] - df[col].median()
            
            # Binary: above/below mean
            df[f'{col}_above_mean'] = (df[col] > df[col].mean()).astype(int)
    
    return df


def engineer_polynomial_features(df, degree=2):
    """Engineer polynomial features for numeric columns"""
    df = df.copy()
    
    numeric_cols = ['height', 'weight']
    existing_cols = [col for col in numeric_cols if col in df.columns]
    
    for col in existing_cols:
        if df[col].notna().sum() > 0:
            for d in range(2, degree + 1):
                df[f'{col}_power_{d}'] = df[col] ** d
    
    # Interaction terms
    if 'height' in df.columns and 'weight' in df.columns:
        df['height_weight_product'] = df['height'] * df['weight']
        df['height_weight_ratio'] = df['height'] / (df['weight'] + 1e-6)
    
    return df


def engineer_sentence_features(df):
    """Engineer features from sentence data if available"""
    df = df.copy()
    
    # Look for sentence-related columns
    sentence_cols = [col for col in df.columns if 'sentence' in col.lower() or 'duration' in col.lower()]
    
    for col in sentence_cols:
        if df[col].notna().sum() > 0:
            # Assuming sentence duration is numeric
            df[f'{col}_log'] = np.log(df[col].fillna(0) + 1)
            df[f'{col}_sqrt'] = np.sqrt(df[col].fillna(0))
            
            # Binary: long vs short sentence
            if df[col].max() > 0:
                median_sentence = df[col].median()
                df[f'{col}_above_median'] = (df[col] > median_sentence).astype(int)
    
    return df


def clean_categorical_feature(series, valid_values=None):
    """Clean and standardize categorical features"""
    series = series.astype(str).str.strip().str.upper()
    series = series.replace(['', 'NAN', 'NONE', 'NULL', 'UNKNOWN', 'NOT AVAILABLE', 'VOID'], 'UNKNOWN')
    
    if valid_values is not None:
        series = series.apply(lambda x: x if x in valid_values else 'UNKNOWN')
    
    return series


def process_person_data(csv_path, config):
    """Load person.csv and apply comprehensive feature engineering"""
    df = pd.read_csv(csv_path, delimiter=';')
    
    print("=== Starting Feature Engineering ===")
    print(f"Initial shape: {df.shape}")
    
    # 1. Body Composition Features
    print("\n[1/9] Engineering body composition features...")
    df = engineer_body_composition_features(df)
    
    # 2. Demographic Features
    print("[2/9] Engineering demographic features...")
    df = engineer_demographic_features(df, config)
    
    # 3. BMI Categories
    print("[3/9] Engineering BMI category features...")
    df = engineer_bmi_categories(df, config)
    
    # 4. Clean categorical features
    print("[4/9] Cleaning categorical features...")
    categorical_features = ['race', 'eyes', 'sex', 'hair']
    for feat in categorical_features:
        if feat in df.columns:
            df[feat] = clean_categorical_feature(df[feat])
    


    df = engineer_categorical_interactions(df)
    


    df = engineer_health_risk_features(df)
    

   
    df = engineer_statistical_features(df)
    

 
    df = engineer_polynomial_features(df, degree=2)
    


    df = engineer_sentence_features(df)
    
    # Remove rows with invalid BMI
    initial_count = len(df)
    df = df.dropna(subset=['bmi'])
    print(f"\nRemoved {initial_count - len(df)} rows with invalid BMI")
    print(f"Final shape after feature engineering: {df.shape}")
    
    # Print feature summary
    print("\n=== Feature Engineering Summary ===")
    print(f"Total features created: {df.shape[1]}")
    print(f"\nFeature categories:")
    
    feature_categories = {
        'Body Composition': [c for c in df.columns if any(x in c for x in ['bmi', 'bsa', 'ponderal', 'weight', 'height'])],
        'Demographic': [c for c in df.columns if any(x in c for x in ['age', 'race', 'sex'])],
        'Health Risk': [c for c in df.columns if any(x in c for x in ['risk', 'metabolic', 'underweight', 'obese'])],
        'Statistical': [c for c in df.columns if any(x in c for x in ['zscore', 'percentile', 'quartile', 'deviation'])],
        'Interactions': [c for c in df.columns if '_' in c and any(x in c for x in ['combo', 'interaction'])],
        'Categorical': [c for c in df.columns if any(x in c for x in ['eyes', 'hair', 'category'])]
    }
    
    for category, features in feature_categories.items():
        if features:
            print(f"  {category}: {len(features)} features")
    
    return df


def normalize_features(df, feature_encoders=None, scalers=None, fit=True):
    """Normalize categorical and numerical features"""
    df = df.copy()
    
    if feature_encoders is None:
        feature_encoders = {}
    if scalers is None:
        scalers = {}
    
    # Categorical encoding
    categorical_cols = [col for col in df.columns if df[col].dtype == 'object' or col.endswith('_category')]
    
    for col in categorical_cols:
        if col in ['name', 'image_path', 'image_stem']:  # Skip ID columns
            continue
            
        if fit:
            le = LabelEncoder()
            try:
                df[f'{col}_encoded'] = le.fit_transform(df[col].fillna('UNKNOWN'))
                feature_encoders[col] = le
            except:
                pass
        else:
            if col in feature_encoders:
                le = feature_encoders[col]
                df[f'{col}_encoded'] = df[col].fillna('UNKNOWN').apply(
                    lambda x: le.transform([x])[0] if x in le.classes_ else -1
                )
    
    # Numerical scaling (for neural networks)
    # CRITICAL: Don't scale BMI, height_m, weight_kg as they need to be in original range for filtering
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    scale_cols = [col for col in numerical_cols if not col.endswith('_encoded') and 
                  col not in ['name', 'image_path', 'bmi', 'height', 'weight', 'height_m', 'weight_kg'] and 
                  not col.startswith('is_') and not col.startswith('has_')]
    
    if fit and len(scale_cols) > 0:
        scaler = RobustScaler()
        df[scale_cols] = scaler.fit_transform(df[scale_cols].fillna(df[scale_cols].median()))
        scalers['numerical'] = scaler
    elif 'numerical' in scalers and len(scale_cols) > 0:
        scaler = scalers['numerical']
        df[scale_cols] = scaler.transform(df[scale_cols].fillna(df[scale_cols].median()))
    
    return df, feature_encoders, scalers


def match_images_to_data(df, image_dirs):
    """Match images to person data and add image path features"""
    
    if isinstance(image_dirs, str):
        image_dirs = [image_dirs]
    
    # Collect all available images
    available_images = {}
    for image_dir in image_dirs:
        if not os.path.exists(image_dir):
            continue
        for f in os.listdir(image_dir):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                stem = os.path.splitext(f)[0].lower()
                available_images[stem] = os.path.join(image_dir, f)
    
    id_columns = ['id', 'ID', 'person_id', 'subject_id', 'doc_id', 'inmate_id']
    id_col = None
    for col in id_columns:
        if col in df.columns:
            id_col = col
            break
    
    if id_col is None:
        id_col = df.columns[0]
    
    df['image_stem'] = df[id_col].astype(str).str.lower()
    df['has_image'] = df['image_stem'].isin(available_images.keys())
    df['image_path'] = df['image_stem'].apply(lambda x: available_images.get(x, None))
    
    matched = df['has_image'].sum()
    print(f"\nMatched {matched}/{len(df)} records to images ({matched/len(df)*100:.1f}%)")
    
    # Filter to only records with images
    df_matched = df[df['has_image']].copy()
    df_matched = df_matched.drop(columns=['name'], errors='ignore')
    df_matched = df_matched.rename(columns={'image_stem': 'name'})
    
    # Engineer image path features
    df_matched = engineer_image_path_features(df_matched)
    
    return df_matched


# ==================== VIT DATA TRANSFORMS ====================
def create_vit_transforms(config):
    """Create training and validation transforms for ViT"""
    
    train_transform = transforms.Compose([
        transforms.Resize((config.image_size, config.image_size)),
        transforms.RandomHorizontalFlip(p=config.horizontal_flip_prob),
        transforms.RandomRotation(degrees=config.rotation_degrees),
        transforms.ColorJitter(
            brightness=config.color_jitter['brightness'],
            contrast=config.color_jitter['contrast'],
            saturation=config.color_jitter['saturation'],
            hue=config.color_jitter['hue']
        ),
        transforms.ToTensor(),
        transforms.Normalize(mean=config.mean, std=config.std)
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((config.image_size, config.image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=config.mean, std=config.std)
    ])
    
    return train_transform, val_transform


# ==================== CUSTOM DATASET CLASS ====================
class BMIDataset(Dataset):
    """PyTorch Dataset for multi-modal BMI prediction with engineered features"""
    
    def __init__(self, dataframe, transform=None, use_features=True, feature_columns=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.use_features = use_features
        
        # Identify available encoded features
        if feature_columns is None:
            self.feature_cols = [col for col in self.df.columns if col.endswith('_encoded')]
        else:
            self.feature_cols = [f'{col}_encoded' for col in feature_columns if f'{col}_encoded' in self.df.columns]
        
        # Identify numerical features to include
        self.numerical_features = [col for col in self.df.columns if 
                                   col.endswith(('_zscore', '_percentile', '_squared', '_log', 
                                                '_interaction', '_ratio', '_score', '_distance',
                                                '_power_2', '_product')) and 
                                   self.df[col].dtype in [np.float32, np.float64, np.int32, np.int64]]
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['name']
        img_path = row['image_path']
        bmi = row['bmi']
        
        try:
            with Image.open(img_path) as image:
                image = image.convert('RGB')
                if self.transform:
                    image = self.transform(image)
        except (IOError, OSError, ValueError):
            size = 224
            if self.transform:
                for t in self.transform.transforms:
                    if hasattr(t, 'size'):
                        size = t.size[0] if isinstance(t.size, tuple) else t.size
                        break
            image = torch.zeros(3, size, size)
        
        sample = {
            'image': image,
            'bmi': torch.tensor(bmi, dtype=torch.float32),
            'image_name': img_name
        }
        
        # Add categorical features
        if self.use_features and len(self.feature_cols) > 0:
            features = torch.tensor([row[col] for col in self.feature_cols], dtype=torch.long)
            sample['categorical_features'] = features
        
        # Add numerical features
        if self.use_features and len(self.numerical_features) > 0:
            numerical_feats = torch.tensor([row[col] for col in self.numerical_features], 
                                          dtype=torch.float32)
            sample['numerical_features'] = numerical_feats
        
        return sample


# ==================== DATA QUALITY CHECKS ====================
def check_data_quality(df):
    """Check for data quality issues"""
    issues = []
    
    missing_bmi = df['bmi'].isna().sum()
    if missing_bmi > 0:
        issues.append(f"Missing BMI values: {missing_bmi}")
    
    duplicate_images = df['name'].duplicated().sum()
    if duplicate_images > 0:
        issues.append(f"Duplicate images: {duplicate_images}")
    
    extreme_outliers = ((df['bmi'] < 10) | (df['bmi'] > 60)).sum()
    if extreme_outliers > 0:
        issues.append(f"Extreme BMI outliers (<10 or >60): {extreme_outliers}")
    
    return len(issues) == 0, issues


# ==================== DATA LOADING AND VALIDATION ====================
def load_and_analyze_dataset(df, show_plots=True):
    """Analyze the BMI dataset with engineered features"""
    is_valid, issues = check_data_quality(df)
    
    if 'bmi' in df.columns:
        fig = plt.figure(figsize=(20, 10))
        
        # BMI Distribution
        plt.subplot(2, 4, 1)
        plt.hist(df['bmi'], bins=50, alpha=0.7, edgecolor='black', color='steelblue')
        plt.xlabel('BMI', fontsize=11)
        plt.ylabel('Frequency', fontsize=11)
        plt.title('BMI Distribution', fontsize=12, fontweight='bold')
        plt.grid(True, alpha=0.3)
        
        # BMI Categories
        plt.subplot(2, 4, 2)
        if 'bmi_category' in df.columns:
            category_counts = df['bmi_category'].value_counts()
            category_counts.plot(kind='bar', color='steelblue', edgecolor='black')
            plt.xlabel('BMI Category', fontsize=11)
            plt.ylabel('Count', fontsize=11)
            plt.title('BMI Categories', fontsize=12, fontweight='bold')
            plt.xticks(rotation=45, ha='right')
            plt.grid(True, alpha=0.3, axis='y')
        
        # Age Distribution
        plt.subplot(2, 4, 3)
        if 'age' in df.columns:
            plt.hist(df['age'].dropna(), bins=30, alpha=0.7, edgecolor='black', color='coral')
            plt.xlabel('Age', fontsize=11)
            plt.ylabel('Frequency', fontsize=11)
            plt.title('Age Distribution', fontsize=12, fontweight='bold')
            plt.grid(True, alpha=0.3)
        
        # BMI vs Age
        plt.subplot(2, 4, 4)
        if 'age' in df.columns:
            plt.scatter(df['age'], df['bmi'], alpha=0.3, s=10)
            plt.xlabel('Age', fontsize=11)
            plt.ylabel('BMI', fontsize=11)
            plt.title('BMI vs Age', fontsize=12, fontweight='bold')
            plt.grid(True, alpha=0.3)
        
        # Height vs Weight
        plt.subplot(2, 4, 5)
        if 'height' in df.columns and 'weight' in df.columns:
            plt.scatter(df['height'], df['weight'], alpha=0.3, s=10, c=df['bmi'], cmap='viridis')
            plt.xlabel('Height (inches)', fontsize=11)
            plt.ylabel('Weight (lbs)', fontsize=11)
            plt.title('Height vs Weight', fontsize=12, fontweight='bold')
            plt.colorbar(label='BMI')
            plt.grid(True, alpha=0.3)
        
        # Sex Distribution
        plt.subplot(2, 4, 6)
        if 'sex' in df.columns:
            sex_counts = df['sex'].value_counts()
            sex_counts.plot(kind='bar', color=['lightblue', 'lightpink'], edgecolor='black')
            plt.xlabel('Sex', fontsize=11)
            plt.ylabel('Count', fontsize=11)
            plt.title('Sex Distribution', fontsize=12, fontweight='bold')
            plt.xticks(rotation=0)
            plt.grid(True, alpha=0.3, axis='y')
        
        # Race Distribution
        plt.subplot(2, 4, 7)
        if 'race' in df.columns:
            race_counts = df['race'].value_counts().head(10)
            race_counts.plot(kind='barh', color='lightgreen', edgecolor='black')
            plt.xlabel('Count', fontsize=11)
            plt.ylabel('Race', fontsize=11)
            plt.title('Top 10 Race Categories', fontsize=12, fontweight='bold')
            plt.grid(True, alpha=0.3, axis='x')
        
        # Statistics Summary
        plt.subplot(2, 4, 8)
        stats_text = f"""
        Dataset Statistics:
        
        Total Samples: {len(df):,}
        
        BMI:
          Mean: {df['bmi'].mean():.2f}
          Std: {df['bmi'].std():.2f}
          Min: {df['bmi'].min():.2f}
          Max: {df['bmi'].max():.2f}
        """
        
        if 'age' in df.columns:
            stats_text += f"""
        Age:
          Mean: {df['age'].mean():.1f}
          Range: {df['age'].min():.0f}-{df['age'].max():.0f}
        """
        
        plt.text(0.1, 0.5, stats_text, fontsize=10, family='monospace',
                verticalalignment='center', transform=plt.gca().transAxes)
        plt.axis('off')
        plt.title('Summary Statistics', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig('illinois_doc_dataset_analysis_enhanced.png', dpi=150, bbox_inches='tight')
        if show_plots:
            plt.show()
        plt.close()
    
    return df


def validate_images(df, min_size=1):
    """Validate image availability and quality"""
    valid_indices = []
    failed_images = []
    
    df = df.reset_index(drop=True)
    
    for idx in tqdm(range(len(df)), total=len(df), desc="Validating images"):
        img_name = df.loc[idx, 'name']
        image_path = df.loc[idx, 'image_path']
        
        if pd.isna(image_path) or image_path is None:
            failed_images.append((idx, img_name, "No path found"))
            continue
        
        if not os.path.exists(image_path):
            failed_images.append((idx, img_name, "File not found"))
            continue
        
        try:
            with Image.open(image_path) as img:
                img.verify()
            
            with Image.open(image_path) as img:
                width, height = img.size
                if width >= min_size and height >= min_size:
                    valid_indices.append(idx)
                else:
                    failed_images.append((idx, img_name, f"Too small: {width}x{height}"))
                    
        except Exception as e:
            failed_images.append((idx, img_name, str(e)))
    
    if failed_images:
        failed_df = pd.DataFrame(failed_images, columns=['index', 'filename', 'reason'])
        failed_df.to_csv('failed_images_log.csv', index=False)
        print(f"\nFailed to validate {len(failed_images)} images. See failed_images_log.csv")
    
    print(f"Successfully validated {len(valid_indices)}/{len(df)} images")
    return valid_indices


def filter_dataset(df, valid_indices, bmi_range=(16, 45)):
    """Filter dataset based on valid images and BMI range"""
    df_filtered = df.iloc[valid_indices].copy()
    
    if 'bmi' in df_filtered.columns:
        initial_len = len(df_filtered)
        df_filtered = df_filtered[
            (df_filtered['bmi'] >= bmi_range[0]) & 
            (df_filtered['bmi'] <= bmi_range[1])
        ].reset_index(drop=True)
        print(f"Filtered {initial_len - len(df_filtered)} samples outside BMI range {bmi_range}")
    
    return df_filtered


# ==================== K-FOLD CROSS VALIDATION SPLITS ====================
def create_kfold_splits(df, config):
    """Create K-fold cross-validation splits with stratification"""
    use_stratified = False
    
    if 'bmi' in df.columns and len(df) >= config.n_folds * 10:
        df_temp = df.copy()
        try:
            df_temp['bmi_quartile'] = pd.qcut(df_temp['bmi'], q=config.n_folds, 
                                             labels=False, duplicates='drop')
            stratify_labels = df_temp['bmi_quartile'].values
            kf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, 
                                random_state=config.random_state)
            split_iterator = kf.split(df, stratify_labels)
            use_stratified = True
            print(f"\nUsing Stratified K-Fold with {config.n_folds} folds")
        except:
            kf = KFold(n_splits=config.n_folds, shuffle=True, 
                      random_state=config.random_state)
            split_iterator = kf.split(df)
            print(f"\nUsing K-Fold with {config.n_folds} folds")
    else:
        kf = KFold(n_splits=config.n_folds, shuffle=True, 
                  random_state=config.random_state)
        split_iterator = kf.split(df)
        print(f"\nUsing K-Fold with {config.n_folds} folds")
    
    fold_splits = []
    for fold, (train_idx, val_idx) in enumerate(split_iterator):
        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df = df.iloc[val_idx].reset_index(drop=True)
        
        # Verify no data leakage
        train_names = set(train_df['name'].astype(str))
        val_names = set(val_df['name'].astype(str))
        assert len(train_names & val_names) == 0, f"Data leakage detected in fold {fold+1}!"
        
        fold_info = {
            'fold': fold + 1,
            'train_size': len(train_df),
            'val_size': len(val_df),
            'train_df': train_df,
            'val_df': val_df
        }
        
        if 'bmi' in df.columns:
            fold_info['train_bmi_stats'] = {
                'mean': float(train_df['bmi'].mean()),
                'std': float(train_df['bmi'].std()),
                'min': float(train_df['bmi'].min()),
                'max': float(train_df['bmi'].max())
            }
            fold_info['val_bmi_stats'] = {
                'mean': float(val_df['bmi'].mean()),
                'std': float(val_df['bmi'].std()),
                'min': float(val_df['bmi'].min()),
                'max': float(val_df['bmi'].max())
            }
        
        fold_splits.append(fold_info)
        
        print(f"  Fold {fold+1}: Train={len(train_df)}, Val={len(val_df)}")
    
    return fold_splits


# ==================== SAVE PREPROCESSED DATA ====================
def save_preprocessed_data(fold_splits, config, image_dirs, feature_encoders, scalers, output_dir='illinois_doc_preprocessed_enhanced'):
    """Save preprocessed fold splits and configuration"""
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\n=== Saving preprocessed data to {output_dir} ===")
    
    # Save configuration
    config_dict = {
        'image_size': config.image_size,
        'channels': config.channels,
        'mean': config.mean,
        'std': config.std,
        'n_folds': config.n_folds,
        'bmi_range': list(config.bmi_range),
        'image_dirs': image_dirs,
        'random_state': config.random_state,
        'feature_columns': config.feature_columns,
        'augmentation': {
            'horizontal_flip_prob': config.horizontal_flip_prob,
            'color_jitter': config.color_jitter,
            'rotation_degrees': config.rotation_degrees
        },
        'age_bins': config.age_bins,
        'age_labels': config.age_labels,
        'bmi_bins': config.bmi_bins,
        'bmi_labels': config.bmi_labels
    }
    
    with open(f'{output_dir}/config.json', 'w') as f:
        json.dump(config_dict, f, indent=4)
    
    # Save feature encoders
    encoder_dict = {}
    for feature, encoder in feature_encoders.items():
        encoder_dict[feature] = {
            'classes': encoder.classes_.tolist()
        }
    
    with open(f'{output_dir}/feature_encoders.json', 'w') as f:
        json.dump(encoder_dict, f, indent=4)
    
    # Save each fold
    for fold_info in fold_splits:
        fold_num = fold_info['fold']
        fold_dir = f'{output_dir}/fold_{fold_num}'
        os.makedirs(fold_dir, exist_ok=True)
        
        fold_info['train_df'].to_csv(f'{fold_dir}/train.csv', index=False)
        fold_info['val_df'].to_csv(f'{fold_dir}/val.csv', index=False)
        
        metadata = {
            'fold': fold_num,
            'train_size': fold_info['train_size'],
            'val_size': fold_info['val_size'],
            'feature_count': len(fold_info['train_df'].columns)
        }
        
        if 'train_bmi_stats' in fold_info:
            metadata['train_bmi_stats'] = fold_info['train_bmi_stats']
            metadata['val_bmi_stats'] = fold_info['val_bmi_stats']
        
        with open(f'{fold_dir}/metadata.json', 'w') as f:
            json.dump(metadata, f, indent=4)
    
    # Save overall summary with feature details
    total_samples = fold_splits[0]['train_size'] + fold_splits[0]['val_size']
    all_features = list(fold_splits[0]['train_df'].columns)
    
    summary = {
        'total_folds': config.n_folds,
        'total_samples': total_samples,
        'total_features': len(all_features),
        'image_size': config.image_size,
        'normalization': {'mean': config.mean, 'std': config.std},
        'feature_columns': all_features,
        'categorical_features': [col for col in all_features if col.endswith('_encoded')],
        'numerical_features': [col for col in all_features if any(x in col for x in 
                              ['_zscore', '_percentile', '_squared', '_log', '_interaction'])],
        'fold_summary': [
            {
                'fold': f['fold'],
                'train_size': f['train_size'],
                'val_size': f['val_size'],
                'train_bmi_mean': f.get('train_bmi_stats', {}).get('mean', 0),
                'val_bmi_mean': f.get('val_bmi_stats', {}).get('mean', 0)
            }
            for f in fold_splits
        ]
    }
    
    with open(f'{output_dir}/summary.json', 'w') as f:
        json.dump(summary, f, indent=4)
    
    print(f"✓ Saved {config.n_folds} folds")
    print(f"✓ Total features: {len(all_features)}")
    print(f"✓ Categorical features: {len(summary['categorical_features'])}")
    print(f"✓ Numerical features: {len(summary['numerical_features'])}")
    
    return output_dir


# ==================== DATALOADER CREATION ====================
def create_dataloaders(fold_dir, config, batch_size=32, num_workers=4, use_features=True):
    """Create PyTorch DataLoaders for a specific fold"""
    train_df = pd.read_csv(f'{fold_dir}/train.csv')
    val_df = pd.read_csv(f'{fold_dir}/val.csv')
    
    train_transform, val_transform = create_vit_transforms(config)
    
    train_dataset = BMIDataset(train_df, transform=train_transform, 
                               use_features=use_features, 
                               feature_columns=config.feature_columns)
    val_dataset = BMIDataset(val_df, transform=val_transform, 
                             use_features=use_features,
                             feature_columns=config.feature_columns)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False
    )
    
    return train_loader, val_loader


# ==================== MAIN PREPROCESSING PIPELINE ====================
def main_preprocessing(person_csv_path, image_dirs, output_dir='illinois_doc_preprocessed_enhanced', show_plots=True):
    """Complete ViT preprocessing pipeline with advanced feature engineering"""
    
    print("=" * 70)
    print("  ILLINOIS DOC DATASET - ADVANCED FEATURE ENGINEERING PIPELINE")
    print("=" * 70)
    
    config = ViTPreprocessingConfig()
    
    # Process person.csv with comprehensive feature engineering
    df = process_person_data(person_csv_path, config)
    
    # Match images to person data
    df_matched = match_images_to_data(df, image_dirs)
    
    if len(df_matched) == 0:
        raise ValueError("No matching images found!")
    
    # Validate images BEFORE filtering (more efficient)
    print("\n=== Validating Images ===")
    valid_indices = validate_images(df_matched, min_size=config.min_image_size)
    
    if len(valid_indices) == 0:
        raise ValueError("No valid images found! Check failed_images_log.csv for details.")
    
    # Filter dataset by BMI range (BEFORE normalization)
    print("\n=== Filtering Dataset by BMI Range ===")
    df_filtered = filter_dataset(df_matched, valid_indices, config.bmi_range)
    
    if len(df_filtered) < config.n_folds * 2:
        raise ValueError(f"Not enough samples ({len(df_filtered)}) for {config.n_folds}-fold CV. Need at least {config.n_folds * 2}")
    
    print(f"✓ Retained {len(df_filtered)} samples within BMI range {config.bmi_range}")
    
    # Normalize features and encode (AFTER filtering, so we fit on the actual training distribution)
    print("\n=== Normalizing Features ===")
    df_normalized, feature_encoders, scalers = normalize_features(df_filtered, fit=True)
    
    # Analyze dataset
    print("\n=== Analyzing Dataset ===")
    df_analyzed = load_and_analyze_dataset(df_normalized, show_plots=show_plots)
    
    # Create K-fold splits
    print("\n=== Creating Cross-Validation Splits ===")
    fold_splits = create_kfold_splits(df_analyzed, config)
    
    # Save preprocessed data
    output_path = save_preprocessed_data(fold_splits, config, image_dirs, 
                                        feature_encoders, scalers, output_dir)
    
    print("\n" + "=" * 70)
    print("  PREPROCESSING COMPLETE!")
    print("=" * 70)
    print(f"✓ Output directory: {output_path}")
    print(f"✓ Ready for training with {len(df_analyzed)} samples")
    print(f"✓ BMI range: {df_analyzed['bmi'].min():.2f} - {df_analyzed['bmi'].max():.2f}")
    
    return output_path, fold_splits, feature_encoders, scalers


# ==================== UTILITY FUNCTIONS ====================
def load_fold_data(fold_dir):
    """Load preprocessed fold data"""
    train_df = pd.read_csv(f'{fold_dir}/train.csv')
    val_df = pd.read_csv(f'{fold_dir}/val.csv')
    
    with open(f'{fold_dir}/metadata.json', 'r') as f:
        metadata = json.load(f)
    
    return train_df, val_df, metadata


def load_config(preprocessed_dir='illinois_doc_preprocessed_enhanced'):
    """Load preprocessing configuration"""
    config_path = f'{preprocessed_dir}/config.json'
    
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Configuration file not found: {config_path}")
    
    with open(config_path, 'r') as f:
        config_dict = json.load(f)
    
    config = ViTPreprocessingConfig()
    config.image_size = config_dict['image_size']
    config.mean = config_dict['mean']
    config.std = config_dict['std']
    config.n_folds = config_dict['n_folds']
    config.bmi_range = tuple(config_dict['bmi_range'])
    config.feature_columns = config_dict.get('feature_columns', [])
    
    if 'augmentation' in config_dict:
        aug = config_dict['augmentation']
        config.horizontal_flip_prob = aug.get('horizontal_flip_prob', 0.5)
        config.color_jitter = aug.get('color_jitter', config.color_jitter)
        config.rotation_degrees = aug.get('rotation_degrees', 10)
    
    if 'age_bins' in config_dict:
        config.age_bins = config_dict['age_bins']
        config.age_labels = config_dict['age_labels']
    
    if 'bmi_bins' in config_dict:
        config.bmi_bins = config_dict['bmi_bins']
        config.bmi_labels = config_dict['bmi_labels']
    
    image_dirs = config_dict.get('image_dirs', [config_dict.get('image_dir', '')])
    
    return config, image_dirs


def load_feature_encoders(preprocessed_dir='illinois_doc_preprocessed_enhanced'):
    """Load feature encoders"""
    encoder_path = f'{preprocessed_dir}/feature_encoders.json'
    
    if not os.path.exists(encoder_path):
        return {}
    
    with open(encoder_path, 'r') as f:
        encoder_dict = json.load(f)
    
    feature_encoders = {}
    for feature, info in encoder_dict.items():
        le = LabelEncoder()
        le.classes_ = np.array(info['classes'])
        feature_encoders[feature] = le
    
    return feature_encoders


def visualize_batch(dataloader, num_images=8, save_path='batch_visualization_enhanced.png'):
    """Visualize a batch of preprocessed images with engineered features"""
    batch = next(iter(dataloader))
    images = batch['image'][:num_images]
    bmis = batch['bmi'][:num_images]
    
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    for idx in range(num_images):
        img = images[idx] * std + mean
        img = torch.clamp(img, 0, 1)
        img = img.permute(1, 2, 0).numpy()
        
        axes[idx].imshow(img)
        
        title = f'BMI: {bmis[idx].item():.2f}'
        
        if 'categorical_features' in batch:
            cat_feats = batch['categorical_features'][idx].tolist()
            title += f'\nCat: {cat_feats[:3]}...'
        
        if 'numerical_features' in batch:
            num_feats = batch['numerical_features'][idx][:3].tolist()
            title += f'\nNum: [{num_feats[0]:.2f}, {num_feats[1]:.2f}, ...]'
        
        axes[idx].set_title(title, fontsize=9, fontweight='bold')
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Batch visualization saved to {save_path}")


def verify_preprocessing(preprocessed_dir='illinois_doc_preprocessed_enhanced'):
    """Verify preprocessing was successful"""
    
    print(f"\n=== Verifying Preprocessing: {preprocessed_dir} ===")
    
    if not os.path.exists(preprocessed_dir):
        print("✗ Preprocessed directory not found")
        return False
    
    if not os.path.exists(f'{preprocessed_dir}/config.json'):
        print("✗ Configuration file not found")
        return False
    
    if not os.path.exists(f'{preprocessed_dir}/feature_encoders.json'):
        print("✗ Feature encoders file not found")
        return False
    
    print("✓ Core files exist")
    
    config, image_dirs = load_config(preprocessed_dir)
    feature_encoders = load_feature_encoders(preprocessed_dir)
    
    print(f"✓ Configuration loaded: {config.n_folds} folds")
    print(f"✓ Feature encoders loaded: {len(feature_encoders)} encoders")
    
    all_folds_ok = True
    for fold in range(1, config.n_folds + 1):
        fold_dir = f'{preprocessed_dir}/fold_{fold}'
        if not os.path.exists(fold_dir):
            print(f"✗ Fold {fold} directory not found")
            all_folds_ok = False
            continue
        
        required_files = ['train.csv', 'val.csv', 'metadata.json']
        for file in required_files:
            if not os.path.exists(f'{fold_dir}/{file}'):
                print(f"✗ Fold {fold}: {file} not found")
                all_folds_ok = False
    
    if all_folds_ok:
        print(f"✓ All {config.n_folds} folds verified")
    
    # Load summary
    if os.path.exists(f'{preprocessed_dir}/summary.json'):
        with open(f'{preprocessed_dir}/summary.json', 'r') as f:
            summary = json.load(f)
        print(f"\n=== Dataset Summary ===")
        print(f"Total samples: {summary['total_samples']}")
        print(f"Total features: {summary['total_features']}")
        print(f"Categorical features: {len(summary.get('categorical_features', []))}")
        print(f"Numerical features: {len(summary.get('numerical_features', []))}")
    
    return all_folds_ok


def print_feature_importance_analysis(df, target='bmi'):
    """Print correlation analysis for engineered features"""
    print("\n=== Feature Correlation Analysis ===")
    
    if target not in df.columns:
        print(f"Target column '{target}' not found")
        return
    
    # Get numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col != target]
    
    if len(numeric_cols) == 0:
        print("No numeric features found")
        return
    
    # Calculate correlations
    correlations = df[numeric_cols].corrwith(df[target]).abs().sort_values(ascending=False)
    
    print(f"\nTop 20 features correlated with {target}:")
    print("-" * 60)
    for i, (feature, corr) in enumerate(correlations.head(20).items(), 1):
        print(f"{i:2d}. {feature:40s} {corr:.4f}")
    
    # Save to file
    correlations.to_csv('feature_correlations.csv', header=['correlation'])
    print(f"\nFull correlation analysis saved to: feature_correlations.csv")


def export_feature_descriptions(output_dir='illinois_doc_preprocessed_enhanced'):
    """Export detailed descriptions of all engineered features"""
    
    feature_descriptions = {
        'Body Composition Features': {
            'bmi': 'Body Mass Index calculated from height and weight',
            'bsa': 'Body Surface Area using DuBois formula',
            'ponderal_index': 'Alternative to BMI: weight/(height^3)',
            'height_m': 'Height converted to meters',
            'weight_kg': 'Weight converted to kilograms',
            'weight_height_ratio': 'Weight to height squared ratio',
            'height_zscore': 'Z-score normalized height',
            'weight_zscore': 'Z-score normalized weight',
            'body_type_score': 'Body type indicator: weight_zscore - height_zscore'
        },
        'Demographic Features': {
            'age': 'Calculated age from date of birth',
            'age_group': 'Age categorized into groups',
            'age_squared': 'Age squared for polynomial relationships',
            'age_decade': 'Age rounded to nearest decade',
            'age_bmi_interaction': 'Interaction between age and BMI',
            'age_weight_interaction': 'Interaction between age and weight'
        },
        'BMI Categories': {
            'bmi_category': 'BMI classified into health categories',
            'is_underweight': 'Binary indicator for BMI < 18.5',
            'is_normal_weight': 'Binary indicator for 18.5 <= BMI < 25',
            'is_overweight': 'Binary indicator for 25 <= BMI < 30',
            'is_obese': 'Binary indicator for BMI >= 30',
            'bmi_distance_from_ideal': 'Absolute distance from ideal BMI (22.5)',
            'bmi_squared': 'BMI squared for polynomial relationships',
            'bmi_log': 'Natural log of BMI',
            'bmi_percentile': 'BMI percentile rank within dataset'
        },
        'Categorical Interactions': {
            'race_sex': 'Interaction between race and sex',
            'race_age': 'Interaction between race and age group',
            'eye_hair_combo': 'Combination of eye and hair color',
            'sex_bmi_category': 'Interaction between sex and BMI category'
        },
        'Health Risk Features': {
            'age_adjusted_bmi_risk': 'BMI risk adjusted for age',
            'metabolic_risk_score': 'Composite metabolic syndrome risk score'
        },
        'Statistical Features': {
            'variable_zscore': 'Z-score normalization of variable',
            'variable_percentile': 'Percentile rank of variable',
            'variable_quartile': 'Quartile assignment of variable',
            'variable_deviation_from_median': 'Distance from median value',
            'variable_above_mean': 'Binary indicator if above mean'
        },
        'Polynomial Features': {
            'height_power_2': 'Height squared',
            'weight_power_2': 'Weight squared',
            'height_weight_product': 'Product of height and weight',
            'height_weight_ratio': 'Ratio of height to weight'
        },
        'Image Features': {
            'image_type': 'Type of mugshot (front, side, inmates)',
            'has_front_image': 'Binary indicator for front view',
            'has_side_image': 'Binary indicator for side view'
        }
    }
    
    # Save to JSON
    with open(f'{output_dir}/feature_descriptions.json', 'w') as f:
        json.dump(feature_descriptions, f, indent=4)
    
    print(f"\nFeature descriptions saved to: {output_dir}/feature_descriptions.json")
    
    # Print summary
    print("\n=== Feature Engineering Summary ===")
    total_features = sum(len(features) for features in feature_descriptions.values())
    print(f"Total feature categories: {len(feature_descriptions)}")
    print(f"Total base features documented: {total_features}")
    
    for category, features in feature_descriptions.items():
        print(f"\n{category}:")
        print(f"  {len(features)} features")


# ==================== EXAMPLE USAGE ====================
if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("  ENHANCED ILLINOIS DOC PREPROCESSING WITH FEATURE ENGINEERING")
    print("=" * 70 + "\n")
    
    # Set paths for Illinois DOC dataset
    person_csv_path = '/kaggle/input/illinois-doc-labeled-faces-dataset/person.csv'
    image_dirs = [
        '/kaggle/input/illinois-doc-labeled-faces-dataset/front/front',
        '/kaggle/input/illinois-doc-labeled-faces-dataset/inmates/inmates',
        '/kaggle/input/illinois-doc-labeled-faces-dataset/side/side'
    ]
    
    # Run preprocessing with feature engineering
    try:
        output_path, fold_splits, feature_encoders, scalers = main_preprocessing(
            person_csv_path, 
            image_dirs, 
            output_dir='/kaggle/working/illinois_doc_preprocessed',  # Match training script path
            show_plots=False  # Set to True in local environment
        )
        
        # Verify preprocessing
        verify_preprocessing(output_path)
        
        # Export feature descriptions
        export_feature_descriptions(output_path)
        
        # Load and analyze first fold
        print("\n=== Testing Fold 1 ===")
        config, image_dirs = load_config(output_path)
        train_loader, val_loader = create_dataloaders(
            f'{output_path}/fold_1', 
            config, 
            batch_size=16,
            num_workers=2,
            use_features=True
        )
        
        # Test one batch
        batch = next(iter(train_loader))
        print(f"\nBatch contents:")
        print(f"  - Images: {batch['image'].shape}")
        print(f"  - BMI: {batch['bmi'].shape}")
        if 'categorical_features' in batch:
            print(f"  - Categorical features: {batch['categorical_features'].shape}")
        if 'numerical_features' in batch:
            print(f"  - Numerical features: {batch['numerical_features'].shape}")
        
        # Visualize batch
        visualize_batch(train_loader, num_images=8, 
                       save_path='illinois_doc_batch_viz_enhanced.png')
        
        # Feature correlation analysis
        train_df = pd.read_csv(f'{output_path}/fold_1/train.csv')
        print_feature_importance_analysis(train_df, target='bmi')
        
        print("\n" + "=" * 70)
        print("  ALL PREPROCESSING STEPS COMPLETED SUCCESSFULLY!")
        print("=" * 70)
        print(f"\nYou can now use the preprocessed data for training:")
        print(f"  - Data location: {output_path}")
        print(f"  - Number of folds: {config.n_folds}")
        print(f"  - Training samples (fold 1): {len(train_df)}")
        print(f"  - Total features: {len(train_df.columns)}")
        
    except Exception as e:
        print(f"\n✗ Error during preprocessing: {str(e)}")
        import traceback
        traceback.print_exc()


  ENHANCED ILLINOIS DOC PREPROCESSING WITH FEATURE ENGINEERING

  ILLINOIS DOC DATASET - ADVANCED FEATURE ENGINEERING PIPELINE
=== Starting Feature Engineering ===
Initial shape: (61110, 22)

[1/9] Engineering body composition features...
[2/9] Engineering demographic features...
[3/9] Engineering BMI category features...
[4/9] Cleaning categorical features...

Removed 395 rows with invalid BMI
Final shape after feature engineering: (60715, 73)

=== Feature Engineering Summary ===
Total features created: 73

Feature categories:
  Body Composition: 38 features
  Demographic: 18 features
  Health Risk: 4 features
  Statistical: 16 features
  Interactions: 3 features
  Categorical: 5 features

Matched 60715/60715 records to images (100.0%)

=== Validating Images ===


Validating images:   0%|          | 0/60715 [00:00<?, ?it/s]


Failed to validate 713 images. See failed_images_log.csv
Successfully validated 60002/60715 images

=== Filtering Dataset by BMI Range ===
Filtered 519 samples outside BMI range (16, 45)
✓ Retained 59483 samples within BMI range (16, 45)

=== Normalizing Features ===

=== Analyzing Dataset ===

=== Creating Cross-Validation Splits ===

Using Stratified K-Fold with 3 folds
  Fold 1: Train=39655, Val=19828
  Fold 2: Train=39655, Val=19828
  Fold 3: Train=39656, Val=19827

=== Saving preprocessed data to /kaggle/working/illinois_doc_preprocessed ===
✓ Saved 3 folds
✓ Total features: 99
✓ Categorical features: 21
✓ Numerical features: 13

  PREPROCESSING COMPLETE!
✓ Output directory: /kaggle/working/illinois_doc_preprocessed
✓ Ready for training with 59483 samples
✓ BMI range: 16.07 - 44.99

=== Verifying Preprocessing: /kaggle/working/illinois_doc_preprocessed ===
✓ Core files exist
✓ Configuration loaded: 3 folds
✓ Feature encoders loaded: 21 encoders
✓ All 3 folds verified

=== Dataset

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import ViTModel


class AdaptivePatchTokenization(nn.Module):
    """Adaptive patch tokenization with learned attention weights"""
    def __init__(self, hidden_dim=768, num_heads=8):
        super(AdaptivePatchTokenization, self).__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        
        # Multi-head attention for patch importance
        self.query = nn.Linear(hidden_dim, hidden_dim)
        self.key = nn.Linear(hidden_dim, hidden_dim)
        self.value = nn.Linear(hidden_dim, hidden_dim)
        
        # Learnable importance scorer
        self.importance_net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, patch_embeddings):
        """
        Args:
            patch_embeddings: [batch_size, num_patches, hidden_dim]
        Returns:
            pooled_output: [batch_size, hidden_dim]
            attention_weights: [batch_size, num_patches]
        """
        batch_size, num_patches, hidden_dim = patch_embeddings.shape
        
        # Compute importance scores for each patch
        importance_scores = self.importance_net(patch_embeddings).squeeze(-1)  # [B, num_patches]
        attention_weights = F.softmax(importance_scores, dim=-1)  # [B, num_patches]
        
        # Multi-head self-attention
        Q = self.query(patch_embeddings).view(batch_size, num_patches, self.num_heads, self.head_dim)
        K = self.key(patch_embeddings).view(batch_size, num_patches, self.num_heads, self.head_dim)
        V = self.value(patch_embeddings).view(batch_size, num_patches, self.num_heads, self.head_dim)
        
        Q = Q.transpose(1, 2)  # [B, num_heads, num_patches, head_dim]
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        
        # Scaled dot-product attention
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.dropout(attn_probs)
        
        attn_output = torch.matmul(attn_probs, V)  # [B, num_heads, num_patches, head_dim]
        attn_output = attn_output.transpose(1, 2).contiguous()  # [B, num_patches, num_heads, head_dim]
        attn_output = attn_output.view(batch_size, num_patches, hidden_dim)
        
        # Project back
        attn_output = self.output_proj(attn_output)
        attn_output = self.dropout(attn_output)
        attn_output = self.layer_norm(attn_output + patch_embeddings)
        
        # Weighted pooling using importance weights
        pooled_output = torch.sum(attn_output * attention_weights.unsqueeze(-1), dim=1)
        
        return pooled_output, attention_weights


class FeatureFusion(nn.Module):
    """Advanced feature fusion for multi-modal inputs"""
    def __init__(self, visual_dim, categorical_dim, numerical_dim, fusion_dim=512):
        super(FeatureFusion, self).__init__()
        
        self.visual_dim = visual_dim
        self.categorical_dim = categorical_dim
        self.numerical_dim = numerical_dim
        
        # Individual projections with normalization
        self.visual_proj = nn.Sequential(
            nn.Linear(visual_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(0.15)
        )
        
        if categorical_dim > 0:
            self.categorical_proj = nn.Sequential(
                nn.Linear(categorical_dim, fusion_dim // 2),
                nn.LayerNorm(fusion_dim // 2),
                nn.GELU(),
                nn.Dropout(0.15)
            )
        
        if numerical_dim > 0:
            self.numerical_proj = nn.Sequential(
                nn.Linear(numerical_dim, fusion_dim // 2),
                nn.LayerNorm(fusion_dim // 2),
                nn.GELU(),
                nn.Dropout(0.15)
            )
        
        # Cross-modal attention
        total_modalities = 1 + (1 if categorical_dim > 0 else 0) + (1 if numerical_dim > 0 else 0)
        fusion_input_dim = fusion_dim + (fusion_dim // 2 if categorical_dim > 0 else 0) + \
                          (fusion_dim // 2 if numerical_dim > 0 else 0)
        
        self.fusion_attention = nn.Sequential(
            nn.Linear(fusion_input_dim, fusion_dim),
            nn.Tanh(),
            nn.Linear(fusion_dim, total_modalities),
            nn.Softmax(dim=-1)
        )
        
        self.output_proj = nn.Sequential(
            nn.Linear(fusion_input_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(0.15)
        )
    
    def forward(self, visual_features, categorical_features=None, numerical_features=None):
        """
        Args:
            visual_features: [batch_size, visual_dim]
            categorical_features: [batch_size, categorical_dim] or None
            numerical_features: [batch_size, numerical_dim] or None
        """
        visual_proj = self.visual_proj(visual_features)
        
        features_list = [visual_proj]
        
        if categorical_features is not None and self.categorical_dim > 0:
            cat_proj = self.categorical_proj(categorical_features)
            features_list.append(cat_proj)
        
        if numerical_features is not None and self.numerical_dim > 0:
            num_proj = self.numerical_proj(numerical_features)
            features_list.append(num_proj)
        
        # Concatenate all features
        fused_features = torch.cat(features_list, dim=-1)
        
        # Apply cross-modal attention
        attention_weights = self.fusion_attention(fused_features)
        
        # Weight and combine
        weighted_features = []
        start_idx = 0
        for i, feat in enumerate(features_list):
            weighted_feat = feat * attention_weights[:, i:i+1]
            weighted_features.append(weighted_feat)
        
        # Final projection
        output = self.output_proj(fused_features)
        
        return output


class RegressionHead(nn.Module):
    """Enhanced regression head with residual connections"""
    def __init__(self, input_dim=512, hidden_dims=[256, 128], dropout=0.2):
        super(RegressionHead, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        self.layers = nn.Sequential(*layers)
        
        # Final prediction layer
        self.output = nn.Linear(prev_dim, 1)
        
        # Residual connection if dimensions match
        self.residual = nn.Linear(input_dim, hidden_dims[-1]) if input_dim != hidden_dims[-1] else nn.Identity()
    
    def forward(self, x):
        residual = self.residual(x)
        out = self.layers(x)
        
        # Add residual before final prediction
        if out.shape[-1] == residual.shape[-1]:
            out = out + residual
        
        return self.output(out)


class ViTForBMI(nn.Module):
    """
    Enhanced ViT model for BMI prediction with:
    - Adaptive patch tokenization
    - Multi-modal feature fusion
    - Advanced regression head
    """
    def __init__(self, num_categorical_features=0, categorical_vocab_sizes=None, 
                 num_numerical_features=0, embed_dim=32, fusion_dim=512,
                 freeze_backbone=True, unfreeze_last_n_layers=4):
        super(ViTForBMI, self).__init__()
        
        # Load pretrained ViT
        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        
        # Freeze backbone if specified
        if freeze_backbone:
            for param in self.vit.parameters():
                param.requires_grad = False
            
            # Unfreeze last N transformer layers
            for layer in self.vit.encoder.layer[-unfreeze_last_n_layers:]:
                for param in layer.parameters():
                    param.requires_grad = True
        
        # Adaptive patch tokenization
        self.adaptive_pooling = AdaptivePatchTokenization(hidden_dim=768, num_heads=8)
        
        # Categorical embeddings
        self.categorical_embeddings = None
        categorical_dim = 0
        if num_categorical_features > 0 and categorical_vocab_sizes:
            self.categorical_embeddings = nn.ModuleList([
                nn.Embedding(vocab_size + 2, embed_dim, padding_idx=0)
                for vocab_size in categorical_vocab_sizes
            ])
            categorical_dim = num_categorical_features * embed_dim
        
        # Numerical feature projection
        self.numerical_proj = None
        numerical_dim = 0
        if num_numerical_features > 0:
            numerical_dim = 128
            self.numerical_proj = nn.Sequential(
                nn.Linear(num_numerical_features, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Dropout(0.15),
                nn.Linear(256, numerical_dim)
            )
        
        # Feature fusion
        self.feature_fusion = FeatureFusion(
            visual_dim=768,
            categorical_dim=categorical_dim,
            numerical_dim=numerical_dim,
            fusion_dim=fusion_dim
        )
        
        # Regression head
        self.regression_head = RegressionHead(
            input_dim=fusion_dim,
            hidden_dims=[256, 128],
            dropout=0.2
        )
        
        # Initialize weights
        self._initialize_weights()
        
        print(f"\n{'='*70}")
        print("MODEL ARCHITECTURE - ViT with Adaptive Patch Tokenization")
        print(f"{'='*70}")
        print(f"Visual Feature Dim:      768 (ViT-Base)")
        print(f"Categorical Features:    {num_categorical_features} -> {categorical_dim}")
        print(f"Numerical Features:      {num_numerical_features} -> {numerical_dim}")
        print(f"Fusion Dim:              {fusion_dim}")
        print(f"Regression Head:         {fusion_dim} -> 256 -> 128 -> 1")
        print(f"Unfrozen ViT Layers:     Last {unfreeze_last_n_layers}")
        print(f"{'='*70}\n")
    
    def _initialize_weights(self):
        """Initialize custom layer weights"""
        for module in [self.adaptive_pooling, self.feature_fusion, self.regression_head]:
            for m in module.modules():
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    if m.bias is not None:
                        nn.init.constant_(m.bias, 0)
                elif isinstance(m, nn.LayerNorm):
                    nn.init.constant_(m.weight, 1)
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, pixel_values, categorical_features=None, numerical_features=None):
        """
        Args:
            pixel_values: [batch_size, 3, 224, 224]
            categorical_features: [batch_size, num_categorical] or None
            numerical_features: [batch_size, num_numerical] or None
        Returns:
            bmi_prediction: [batch_size, 1]
        """
        # Extract ViT features
        outputs = self.vit(pixel_values=pixel_values)
        last_hidden_state = outputs.last_hidden_state  # [B, num_patches+1, 768]
        
        # Remove CLS token, keep only patch tokens
        patch_tokens = last_hidden_state[:, 1:, :]  # [B, num_patches, 768]
        
        # Adaptive patch tokenization
        visual_features, patch_weights = self.adaptive_pooling(patch_tokens)  # [B, 768]
        
        # Process categorical features
        cat_features = None
        if categorical_features is not None and self.categorical_embeddings is not None:
            cat_embeds = []
            for i, embedding_layer in enumerate(self.categorical_embeddings):
                feat = categorical_features[:, i] + 1  # Shift for padding
                max_idx = embedding_layer.num_embeddings - 1
                feat = torch.clamp(feat, min=0, max=max_idx)
                emb = embedding_layer(feat)
                cat_embeds.append(emb)
            cat_features = torch.cat(cat_embeds, dim=-1)  # [B, categorical_dim]
        
        # Process numerical features
        num_features = None
        if numerical_features is not None and self.numerical_proj is not None:
            numerical_features = torch.nan_to_num(numerical_features, nan=0.0)
            num_features = self.numerical_proj(numerical_features)  # [B, numerical_dim]
        
        # Feature fusion
        fused_features = self.feature_fusion(visual_features, cat_features, num_features)
        
        # Regression
        bmi_prediction = self.regression_head(fused_features)
        
        return bmi_prediction


def count_parameters(model):
    """Count trainable and total parameters"""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params
    
    print(f"\nModel Parameter Summary:")
    print(f"{'='*50}")
    print(f"Total Parameters:      {total_params:,}")
    print(f"Trainable Parameters:  {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")
    print(f"Frozen Parameters:     {frozen_params:,} ({frozen_params/total_params*100:.2f}%)")
    print(f"{'='*50}\n")
    
    return total_params, trainable_params


if __name__ == "__main__":
    # Test model creation
    print("Testing model creation...\n")
    
    model = ViTForBMI(
        num_categorical_features=4,
        categorical_vocab_sizes=[10, 5, 2, 8],
        num_numerical_features=15,
        embed_dim=32,
        fusion_dim=512
    )
    
    count_parameters(model)
    
    # Test forward pass
    batch_size = 4
    pixel_values = torch.randn(batch_size, 3, 224, 224)
    categorical_features = torch.randint(0, 5, (batch_size, 4))
    numerical_features = torch.randn(batch_size, 15)
    
    output = model(pixel_values, categorical_features, numerical_features)
    
    print(f"Input shapes:")
    print(f"  Images: {pixel_values.shape}")
    print(f"  Categorical: {categorical_features.shape}")
    print(f"  Numerical: {numerical_features.shape}")
    print(f"\nOutput shape: {output.shape}")
    print(f"\n✓ Model test passed!")

import os
import sys
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts


# For loading preprocessed data
from sklearn.preprocessing import LabelEncoder


# ==================== CONFIGURATION ====================
class TrainingConfig:
    """Training configuration"""
    def __init__(self):
        # Paths
        self.preprocessed_dir = '/kaggle/working/illinois_doc_preprocessed'
        self.output_dir = '/kaggle/working/vit_adaptive_checkpoints'
        self.log_dir = '/kaggle/working/logs'
        
        # Training
        self.n_folds = 3
        self.epochs = 30
        self.batch_size = 48
        self.accumulation_steps = 1  # For gradient accumulation
        
        # Optimizer
        self.lr = 3e-5
        self.vit_lr_multiplier = 0.1  # Lower LR for pretrained ViT
        self.weight_decay = 0.01
        self.max_grad_norm = 1.0
        
        # Scheduler
        self.warmup_pct = 0.1
        self.scheduler_type = 'onecycle'  # 'onecycle' or 'cosine'
        
        # Model
        self.embed_dim = 32
        self.fusion_dim = 512
        self.unfreeze_last_n_layers = 4
        
        # Loss
        self.loss_type = 'huber'  # 'mse', 'mae', 'huber', 'smooth_l1'
        self.huber_delta = 1.0
        
        # Regularization
        self.dropout = 0.2
        self.label_smoothing = 0.0
        
        # Early stopping
        self.patience = 7
        self.min_delta = 0.0001
        
        # Data loading
        self.num_workers = 2
        self.pin_memory = True
        self.prefetch_factor = 2
        
        # Mixed precision
        self.use_amp = True
        
        # Validation frequency
        self.val_every_n_epochs = 1
        
        # Seed
        self.seed = 42


# ==================== UTILS ====================
def set_seed(seed=42):
    """Set random seeds for reproducibility"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_device():
    """Get available device"""
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    else:
        device = torch.device('cpu')
        print("⚠ Using CPU")
    return device


def load_config_and_encoders(preprocessed_dir):
    """Load preprocessing config and feature encoders"""
    with open(f'{preprocessed_dir}/config.json', 'r') as f:
        config_dict = json.load(f)
    
    with open(f'{preprocessed_dir}/feature_encoders.json', 'r') as f:
        encoder_dict = json.load(f)
    
    # Reconstruct encoders
    feature_encoders = {}
    for feature, info in encoder_dict.items():
        le = LabelEncoder()
        le.classes_ = np.array(info['classes'])
        feature_encoders[feature] = le
    
    return config_dict, feature_encoders


def get_categorical_vocab_sizes(feature_encoders):
    """Get vocabulary sizes for categorical features"""
    vocab_sizes = []
    for feat_name, encoder in feature_encoders.items():
        if not feat_name.endswith('_encoded'):
            vocab_sizes.append(len(encoder.classes_))
    return vocab_sizes


# ==================== DATA LOADING ====================
class BMIDataset(torch.utils.data.Dataset):
    """Dataset for BMI prediction"""
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        
        # Identify feature columns
        self.categorical_cols = [col for col in self.df.columns if col.endswith('_encoded')]
        self.numerical_cols = [col for col in self.df.columns if 
                              col.endswith(('_zscore', '_percentile', '_squared', '_log', 
                                          '_interaction', '_ratio', '_score', '_distance',
                                          '_power_2', '_product')) and 
                              self.df[col].dtype in [np.float32, np.float64, np.int32, np.int64]]
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        from PIL import Image
        img_path = row['image_path']
        try:
            with Image.open(img_path) as image:
                image = image.convert('RGB')
                if self.transform:
                    image = self.transform(image)
        except:
            # Return zero tensor if image fails
            image = torch.zeros(3, 224, 224)
        
        # Get BMI target
        bmi = torch.tensor(row['bmi'], dtype=torch.float32)
        
        # Get categorical features
        categorical_features = None
        if len(self.categorical_cols) > 0:
            categorical_features = torch.tensor([row[col] for col in self.categorical_cols], 
                                                dtype=torch.long)
        
        # Get numerical features
        numerical_features = None
        if len(self.numerical_cols) > 0:
            numerical_features = torch.tensor([row[col] for col in self.numerical_cols], 
                                             dtype=torch.float32)
        
        return {
            'image': image,
            'bmi': bmi,
            'categorical_features': categorical_features,
            'numerical_features': numerical_features,
            'image_name': row['name']
        }


def get_transforms(config_dict):
    """Create data transforms"""
    from torchvision import transforms
    
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean=config_dict['mean'], std=config_dict['std'])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=config_dict['mean'], std=config_dict['std'])
    ])
    
    return train_transform, val_transform


def get_dataloaders(fold, config, preprocess_config):
    """Create train and validation dataloaders"""
    train_transform, val_transform = get_transforms(preprocess_config)
    
    # Load data
    train_df = pd.read_csv(f'{config.preprocessed_dir}/fold_{fold}/train.csv')
    val_df = pd.read_csv(f'{config.preprocessed_dir}/fold_{fold}/val.csv')
    
    print(f"\nFold {fold} dataset sizes:")
    print(f"  Train: {len(train_df):,} samples")
    print(f"  Val:   {len(val_df):,} samples")
    
    # Create datasets
    train_dataset = BMIDataset(train_df, transform=train_transform)
    val_dataset = BMIDataset(val_df, transform=val_transform)
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=config.pin_memory,
        prefetch_factor=config.prefetch_factor,
        persistent_workers=True,
        drop_last=True  # Drop last incomplete batch
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size * 2,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=config.pin_memory,
        prefetch_factor=config.prefetch_factor,
        persistent_workers=True
    )
    
    return train_loader, val_loader


# ==================== LOSS FUNCTIONS ====================
def get_loss_function(config):
    """Get loss function based on config"""
    if config.loss_type == 'mse':
        return nn.MSELoss()
    elif config.loss_type == 'mae':
        return nn.L1Loss()
    elif config.loss_type == 'huber':
        return nn.HuberLoss(delta=config.huber_delta)
    elif config.loss_type == 'smooth_l1':
        return nn.SmoothL1Loss()
    else:
        raise ValueError(f"Unknown loss type: {config.loss_type}")


# ==================== METRICS ====================
def calculate_metrics(predictions, targets):
    """Calculate regression metrics"""
    predictions = np.array(predictions).flatten()
    targets = np.array(targets).flatten()
    
    mae = np.mean(np.abs(predictions - targets))
    mse = np.mean((predictions - targets) ** 2)
    rmse = np.sqrt(mse)
    
    # R² score
    ss_res = np.sum((targets - predictions) ** 2)
    ss_tot = np.sum((targets - np.mean(targets)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
    
    return {
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'r2': r2
    }


# ==================== TRAINING ====================
class EarlyStopping:
    """Early stopping handler"""
    def __init__(self, patience=7, min_delta=0.0001, mode='min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0
    
    def __call__(self, score, epoch):
        if self.best_score is None:
            self.best_score = score
            self.best_epoch = epoch
            return False
        
        if self.mode == 'min':
            improved = score < (self.best_score - self.min_delta)
        else:
            improved = score > (self.best_score + self.min_delta)
        
        if improved:
            self.best_score = score
            self.best_epoch = epoch
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return False


def train_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, 
                device, config, epoch):
    """Train for one epoch"""
    model.train()
    
    total_loss = 0
    num_batches = len(train_loader)
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}', leave=False)
    
    optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(pbar):
        # Move to device
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['bmi'].to(device, non_blocking=True)
        
        categorical_features = batch.get('categorical_features')
        if categorical_features is not None:
            categorical_features = categorical_features.to(device, non_blocking=True)
        
        numerical_features = batch.get('numerical_features')
        if numerical_features is not None:
            numerical_features = numerical_features.to(device, non_blocking=True)
        
        # Forward pass with mixed precision
        with autocast(enabled=config.use_amp):
            predictions = model(images, categorical_features, numerical_features)
            loss = criterion(predictions.squeeze(), targets)
            
            # Scale loss for gradient accumulation
            loss = loss / config.accumulation_steps
        
        # Backward pass
        scaler.scale(loss).backward()
        
        # Update weights every accumulation_steps
        if (batch_idx + 1) % config.accumulation_steps == 0:
            # Gradient clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
            
            # Optimizer step
            scaler.step(optimizer)
            scaler.update()
            
            # Scheduler step
            if scheduler is not None:
                scheduler.step()
            
            optimizer.zero_grad()
        
        # Track loss
        total_loss += loss.item() * config.accumulation_steps
        
        # Update progress bar
        pbar.set_postfix({'loss': f'{total_loss / (batch_idx + 1):.4f}'})
    
    return total_loss / num_batches


@torch.no_grad()
def validate(model, val_loader, criterion, device, config):
    """Validate model"""
    model.eval()
    
    total_loss = 0
    all_predictions = []
    all_targets = []
    
    pbar = tqdm(val_loader, desc='Validating', leave=False)
    
    for batch in pbar:
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['bmi'].to(device, non_blocking=True)
        
        categorical_features = batch.get('categorical_features')
        if categorical_features is not None:
            categorical_features = categorical_features.to(device, non_blocking=True)
        
        numerical_features = batch.get('numerical_features')
        if numerical_features is not None:
            numerical_features = numerical_features.to(device, non_blocking=True)
        
        # Forward pass
        with autocast(enabled=config.use_amp):
            predictions = model(images, categorical_features, numerical_features)
            loss = criterion(predictions.squeeze(), targets)
        
        total_loss += loss.item()
        
        # Collect predictions and targets
        all_predictions.extend(predictions.cpu().numpy().flatten())
        all_targets.extend(targets.cpu().numpy().flatten())
    
    avg_loss = total_loss / len(val_loader)
    metrics = calculate_metrics(all_predictions, all_targets)
    metrics['loss'] = avg_loss
    
    return metrics


def save_checkpoint(model, optimizer, scheduler, epoch, metrics, fold, config):
    """Save model checkpoint"""
    os.makedirs(config.output_dir, exist_ok=True)
    
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'metrics': metrics,
        'config': config.__dict__
    }
    
    path = f'{config.output_dir}/fold_{fold}_best.pt'
    torch.save(checkpoint, path)
    
    return path


def load_checkpoint(path, model, optimizer=None, scheduler=None):
    """Load model checkpoint"""
    # Load with weights_only=False to allow numpy objects in metrics
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    
    if optimizer and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    if scheduler and 'scheduler_state_dict' in checkpoint and checkpoint['scheduler_state_dict']:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    return checkpoint['epoch'], checkpoint['metrics']


# ==================== TRAINING LOOP ====================
def train_fold(fold, config, device):
    """Train model for one fold"""
    print(f"\n{'='*70}")
    print(f"TRAINING FOLD {fold}/{config.n_folds}")
    print(f"{'='*70}\n")
    
    # Load preprocessing config and encoders
    preprocess_config, feature_encoders = load_config_and_encoders(config.preprocessed_dir)
    
    # Get dataloaders
    train_loader, val_loader = get_dataloaders(fold, config, preprocess_config)
    
    # Get feature dimensions from first batch
    sample_batch = next(iter(train_loader))
    num_categorical = sample_batch['categorical_features'].shape[1] if sample_batch['categorical_features'] is not None else 0
    num_numerical = sample_batch['numerical_features'].shape[1] if sample_batch['numerical_features'] is not None else 0
    categorical_vocab_sizes = get_categorical_vocab_sizes(feature_encoders)
    
    print(f"\nFeature configuration:")
    print(f"  Categorical features: {num_categorical}")
    print(f"  Numerical features:   {num_numerical}")
    print(f"  Vocabulary sizes:     {categorical_vocab_sizes}\n")
    
    # Create model
    model = ViTForBMI(
        num_categorical_features=num_categorical,
        categorical_vocab_sizes=categorical_vocab_sizes,
        num_numerical_features=num_numerical,
        embed_dim=config.embed_dim,
        fusion_dim=config.fusion_dim,
        unfreeze_last_n_layers=config.unfreeze_last_n_layers
    ).to(device)
    
    # Count parameters
    count_parameters(model)
    
    # Get loss function
    criterion = get_loss_function(config)
    print(f"Loss function: {config.loss_type}")
    
    # Create optimizer with different learning rates
    vit_params = [p for n, p in model.named_parameters() if 'vit' in n and p.requires_grad]
    other_params = [p for n, p in model.named_parameters() if 'vit' not in n and p.requires_grad]
    
    optimizer = AdamW([
        {'params': vit_params, 'lr': config.lr * config.vit_lr_multiplier},
        {'params': other_params, 'lr': config.lr}
    ], weight_decay=config.weight_decay)
    
    # Create scheduler
    if config.scheduler_type == 'onecycle':
        scheduler = OneCycleLR(
            optimizer,
            max_lr=[config.lr * config.vit_lr_multiplier, config.lr],
            epochs=config.epochs,
            steps_per_epoch=len(train_loader) // config.accumulation_steps,
            pct_start=config.warmup_pct,
            anneal_strategy='cos'
        )
    elif config.scheduler_type == 'cosine':
        scheduler = CosineAnnealingWarmRestarts(
            optimizer,
            T_0=len(train_loader) // config.accumulation_steps * 5,
            T_mult=1
        )
    else:
        scheduler = None
    
    # Mixed precision scaler
    scaler = GradScaler(enabled=config.use_amp)
    
    # Early stopping
    early_stopping = EarlyStopping(patience=config.patience, min_delta=config.min_delta)
    
    # Training history
    history = {
        'train_losses': [],
        'val_losses': [],
        'val_maes': [],
        'val_mses': [],
        'val_rmses': [],
        'val_r2s': []
    }
    
    best_mae = float('inf')
    best_metrics = None
    
    print(f"\nStarting training...")
    print(f"{'='*70}\n")
    
    start_time = time.time()
    
    for epoch in range(1, config.epochs + 1):
        epoch_start = time.time()
        
        # Train
        train_loss = train_epoch(model, train_loader, criterion, optimizer, 
                                scheduler, scaler, device, config, epoch)
        history['train_losses'].append(train_loss)
        
        # Validate
        if epoch % config.val_every_n_epochs == 0 or epoch == config.epochs:
            val_metrics = validate(model, val_loader, criterion, device, config)
            
            history['val_losses'].append(val_metrics['loss'])
            history['val_maes'].append(val_metrics['mae'])
            history['val_mses'].append(val_metrics['mse'])
            history['val_rmses'].append(val_metrics['rmse'])
            history['val_r2s'].append(val_metrics['r2'])
            
            epoch_time = time.time() - epoch_start
            
            print(f"Epoch {epoch:02d}/{config.epochs} | "
                  f"Time: {epoch_time:.1f}s | "
                  f"Train Loss: {train_loss:.4f} | "
                  f"Val Loss: {val_metrics['loss']:.4f} | "
                  f"MAE: {val_metrics['mae']:.4f} | "
                  f"RMSE: {val_metrics['rmse']:.4f} | "
                  f"R²: {val_metrics['r2']:.4f}")
            
            # Save best model
            if val_metrics['mae'] < best_mae:
                best_mae = val_metrics['mae']
                best_metrics = val_metrics.copy()
                checkpoint_path = save_checkpoint(model, optimizer, scheduler, 
                                                 epoch, val_metrics, fold, config)
                print(f"  ✓ New best model saved (MAE: {best_mae:.4f})")
            
            # Early stopping check
            early_stopping(val_metrics['mae'], epoch)
            if early_stopping.early_stop:
                print(f"\n  Early stopping triggered at epoch {epoch}")
                print(f"  Best epoch was {early_stopping.best_epoch}")
                break
        else:
            epoch_time = time.time() - epoch_start
            print(f"Epoch {epoch:02d}/{config.epochs} | "
                  f"Time: {epoch_time:.1f}s | "
                  f"Train Loss: {train_loss:.4f}")
    
    total_time = time.time() - start_time
    
    # Load best model for final evaluation
    checkpoint_path = f'{config.output_dir}/fold_{fold}_best.pt'
    if os.path.exists(checkpoint_path):
        _, _ = load_checkpoint(checkpoint_path, model)
        final_metrics = validate(model, val_loader, criterion, device, config)
    else:
        final_metrics = best_metrics
    
    print(f"\n{'='*70}")
    print(f"FOLD {fold} RESULTS")
    print(f"{'='*70}")
    print(f"Training time:  {total_time/60:.2f} minutes")
    print(f"MAE:            {final_metrics['mae']:.4f}")
    print(f"MSE:            {final_metrics['mse']:.4f}")
    print(f"RMSE:           {final_metrics['rmse']:.4f}")
    print(f"R²:             {final_metrics['r2']:.4f}")
    print(f"{'='*70}\n")
    
    # Add final metrics to history
    history['final_metrics'] = final_metrics
    history['training_time'] = total_time
    
    return history


# ==================== VISUALIZATION ====================
def plot_training_history(all_histories, config):
    """Plot training history for all folds"""
    os.makedirs(config.log_dir, exist_ok=True)
    
    fig = plt.figure(figsize=(20, 12))
    
    # Training loss
    ax1 = plt.subplot(3, 3, 1)
    for fold_idx, history in enumerate(all_histories):
        epochs = range(1, len(history['train_losses']) + 1)
        ax1.plot(epochs, history['train_losses'], marker='o', 
                label=f'Fold {fold_idx+1}', linewidth=2, markersize=4)
    ax1.set_xlabel('Epoch', fontweight='bold')
    ax1.set_ylabel('Loss', fontweight='bold')
    ax1.set_title('Training Loss', fontweight='bold', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Validation loss
    ax2 = plt.subplot(3, 3, 2)
    for fold_idx, history in enumerate(all_histories):
        val_epochs = np.arange(config.val_every_n_epochs, 
                               len(history['val_losses']) * config.val_every_n_epochs + 1, 
                               config.val_every_n_epochs)
        ax2.plot(val_epochs, history['val_losses'], marker='s', 
                label=f'Fold {fold_idx+1}', linewidth=2, markersize=4)
    ax2.set_xlabel('Epoch', fontweight='bold')
    ax2.set_ylabel('Loss', fontweight='bold')
    ax2.set_title('Validation Loss', fontweight='bold', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # MAE
    ax3 = plt.subplot(3, 3, 3)
    for fold_idx, history in enumerate(all_histories):
        val_epochs = np.arange(config.val_every_n_epochs, 
                               len(history['val_maes']) * config.val_every_n_epochs + 1, 
                               config.val_every_n_epochs)
        ax3.plot(val_epochs, history['val_maes'], marker='^', 
                label=f'Fold {fold_idx+1}', linewidth=2, markersize=4)
    ax3.set_xlabel('Epoch', fontweight='bold')
    ax3.set_ylabel('MAE', fontweight='bold')
    ax3.set_title('Validation MAE', fontweight='bold', fontsize=12)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # RMSE
    ax4 = plt.subplot(3, 3, 4)
    for fold_idx, history in enumerate(all_histories):
        val_epochs = np.arange(config.val_every_n_epochs, 
                               len(history['val_rmses']) * config.val_every_n_epochs + 1, 
                               config.val_every_n_epochs)
        ax4.plot(val_epochs, history['val_rmses'], marker='d', 
                label=f'Fold {fold_idx+1}', linewidth=2, markersize=4)
    ax4.set_xlabel('Epoch', fontweight='bold')
    ax4.set_ylabel('RMSE', fontweight='bold')
    ax4.set_title('Validation RMSE', fontweight='bold', fontsize=12)
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # R² Score
    ax5 = plt.subplot(3, 3, 5)
    for fold_idx, history in enumerate(all_histories):
        val_epochs = np.arange(config.val_every_n_epochs, 
                               len(history['val_r2s']) * config.val_every_n_epochs + 1, 
                               config.val_every_n_epochs)
        ax5.plot(val_epochs, history['val_r2s'], marker='*', 
                label=f'Fold {fold_idx+1}', linewidth=2, markersize=6)
    ax5.set_xlabel('Epoch', fontweight='bold')
    ax5.set_ylabel('R²', fontweight='bold')
    ax5.set_title('Validation R²', fontweight='bold', fontsize=12)
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # Final metrics comparison
    fold_labels = [f'Fold {i+1}' for i in range(len(all_histories))]
    
    # Final MAE
    ax6 = plt.subplot(3, 3, 6)
    maes = [h['final_metrics']['mae'] for h in all_histories]
    bars = ax6.bar(fold_labels, maes, color='steelblue', edgecolor='black', alpha=0.7)
    ax6.axhline(np.mean(maes), color='red', linestyle='--', linewidth=2, 
               label=f'Mean: {np.mean(maes):.4f}')
    for bar, val in zip(bars, maes):
        ax6.text(bar.get_x() + bar.get_width()/2., bar.get_height(), 
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
    ax6.set_ylabel('MAE', fontweight='bold')
    ax6.set_title('Final MAE by Fold', fontweight='bold', fontsize=12)
    ax6.legend()
    ax6.grid(True, alpha=0.3, axis='y')
    
    # Final RMSE
    ax7 = plt.subplot(3, 3, 7)
    rmses = [h['final_metrics']['rmse'] for h in all_histories]
    bars = ax7.bar(fold_labels, rmses, color='coral', edgecolor='black', alpha=0.7)
    ax7.axhline(np.mean(rmses), color='red', linestyle='--', linewidth=2, 
               label=f'Mean: {np.mean(rmses):.4f}')
    for bar, val in zip(bars, rmses):
        ax7.text(bar.get_x() + bar.get_width()/2., bar.get_height(), 
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
    ax7.set_ylabel('RMSE', fontweight='bold')
    ax7.set_title('Final RMSE by Fold', fontweight='bold', fontsize=12)
    ax7.legend()
    ax7.grid(True, alpha=0.3, axis='y')
    
    # Final R²
    ax8 = plt.subplot(3, 3, 8)
    r2s = [h['final_metrics']['r2'] for h in all_histories]
    bars = ax8.bar(fold_labels, r2s, color='lightgreen', edgecolor='black', alpha=0.7)
    ax8.axhline(np.mean(r2s), color='red', linestyle='--', linewidth=2, 
               label=f'Mean: {np.mean(r2s):.4f}')
    for bar, val in zip(bars, r2s):
        ax8.text(bar.get_x() + bar.get_width()/2., bar.get_height(), 
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
    ax8.set_ylabel('R²', fontweight='bold')
    ax8.set_title('Final R² by Fold', fontweight='bold', fontsize=12)
    ax8.legend()
    ax8.grid(True, alpha=0.3, axis='y')
    
    # Training time
    ax9 = plt.subplot(3, 3, 9)
    times = [h['training_time']/60 for h in all_histories]
    bars = ax9.bar(fold_labels, times, color='plum', edgecolor='black', alpha=0.7)
    for bar, val in zip(bars, times):
        ax9.text(bar.get_x() + bar.get_width()/2., bar.get_height(), 
                f'{val:.1f}m', ha='center', va='bottom', fontweight='bold')
    ax9.set_ylabel('Time (minutes)', fontweight='bold')
    ax9.set_title('Training Time by Fold', fontweight='bold', fontsize=12)
    ax9.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plot_path = f'{config.log_dir}/training_results.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Training plots saved to: {plot_path}")

def save_results(all_histories, config):
    """Save training results to JSON"""
    os.makedirs(config.log_dir, exist_ok=True)
    
    # Helper function to convert numpy types to native Python types
    def convert_to_native(obj):
        """Convert numpy types to native Python types"""
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {key: convert_to_native(value) for key, value in obj.items()}
        elif isinstance(obj, list):
            return [convert_to_native(item) for item in obj]
        else:
            return obj
    
    results = {
        'config': config.__dict__,
        'folds': []
    }
    
    for fold_idx, history in enumerate(all_histories):
        fold_results = {
            'fold': fold_idx + 1,
            'final_metrics': convert_to_native(history['final_metrics']),
            'training_time': float(history['training_time']),
            'num_epochs': len(history['train_losses'])
        }
        results['folds'].append(fold_results)
    
    # Compute aggregate statistics
    results['aggregate'] = {
        'mae': {
            'mean': float(np.mean([h['final_metrics']['mae'] for h in all_histories])),
            'std': float(np.std([h['final_metrics']['mae'] for h in all_histories])),
            'min': float(np.min([h['final_metrics']['mae'] for h in all_histories])),
            'max': float(np.max([h['final_metrics']['mae'] for h in all_histories]))
        },
        'rmse': {
            'mean': float(np.mean([h['final_metrics']['rmse'] for h in all_histories])),
            'std': float(np.std([h['final_metrics']['rmse'] for h in all_histories])),
            'min': float(np.min([h['final_metrics']['rmse'] for h in all_histories])),
            'max': float(np.max([h['final_metrics']['rmse'] for h in all_histories]))
        },
        'r2': {
            'mean': float(np.mean([h['final_metrics']['r2'] for h in all_histories])),
            'std': float(np.std([h['final_metrics']['r2'] for h in all_histories])),
            'min': float(np.min([h['final_metrics']['r2'] for h in all_histories])),
            'max': float(np.max([h['final_metrics']['r2'] for h in all_histories]))
        },
        'total_training_time': float(sum([h['training_time'] for h in all_histories]))
    }
    
    results_path = f'{config.log_dir}/results.json'
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=4)
    
    print(f"✓ Results saved to: {results_path}")


# ==================== MAIN ====================
def main():
    """Main training pipeline"""
    print("\n" + "="*70)
    print("  ViT WITH ADAPTIVE PATCH TOKENIZATION - BMI PREDICTION")
    print("="*70 + "\n")
    
    # Configuration
    config = TrainingConfig()
    
    # Set seed
    set_seed(config.seed)
    
    # Get device
    device = get_device()
    
    print(f"\nConfiguration:")
    print(f"  Folds:          {config.n_folds}")
    print(f"  Epochs:         {config.epochs}")
    print(f"  Batch size:     {config.batch_size}")
    print(f"  Learning rate:  {config.lr}")
    print(f"  Loss function:  {config.loss_type}")
    print(f"  Mixed precision: {config.use_amp}")
    print(f"  Output dir:     {config.output_dir}\n")
    
    # Train all folds
    all_histories = []
    
    for fold in range(1, config.n_folds + 1):
        history = train_fold(fold, config, device)
        all_histories.append(history)
    
    # Plot results
    print(f"\n{'='*70}")
    print("GENERATING PLOTS AND SAVING RESULTS")
    print(f"{'='*70}\n")
    
    plot_training_history(all_histories, config)
    save_results(all_histories, config)
    
    # Print final summary
    print(f"\n{'='*70}")
    print("FINAL CROSS-VALIDATION RESULTS")
    print(f"{'='*70}")
    print(f"{'Metric':<10} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
    print("-"*70)
    
    maes = [h['final_metrics']['mae'] for h in all_histories]
    rmses = [h['final_metrics']['rmse'] for h in all_histories]
    r2s = [h['final_metrics']['r2'] for h in all_histories]
    
    print(f"{'MAE':<10} {np.mean(maes):<12.4f} {np.std(maes):<12.4f} "
          f"{np.min(maes):<12.4f} {np.max(maes):<12.4f}")
    print(f"{'RMSE':<10} {np.mean(rmses):<12.4f} {np.std(rmses):<12.4f} "
          f"{np.min(rmses):<12.4f} {np.max(rmses):<12.4f}")
    print(f"{'R²':<10} {np.mean(r2s):<12.4f} {np.std(r2s):<12.4f} "
          f"{np.min(r2s):<12.4f} {np.max(r2s):<12.4f}")
    
    total_time = sum([h['training_time'] for h in all_histories])
    print(f"\nTotal training time: {total_time/60:.2f} minutes ({total_time/3600:.2f} hours)")
    print(f"{'='*70}\n")
    
    print("✓ Training complete!")
    print(f"✓ Models saved in: {config.output_dir}")
    print(f"✓ Logs saved in: {config.log_dir}")


if __name__ == "__main__":
    main()

2025-10-13 13:40:59.438301: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760362859.612136      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760362859.667270      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Testing model creation...



config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]


MODEL ARCHITECTURE - ViT with Adaptive Patch Tokenization
Visual Feature Dim:      768 (ViT-Base)
Categorical Features:    4 -> 128
Numerical Features:      15 -> 128
Fusion Dim:              512
Regression Head:         512 -> 256 -> 128 -> 1
Unfrozen ViT Layers:     Last 4


Model Parameter Summary:
Total Parameters:      90,832,933
Trainable Parameters:  32,795,173 (36.10%)
Frozen Parameters:     58,037,760 (63.90%)

Input shapes:
  Images: torch.Size([4, 3, 224, 224])
  Categorical: torch.Size([4, 4])
  Numerical: torch.Size([4, 15])

Output shape: torch.Size([4, 1])

✓ Model test passed!

  ViT WITH ADAPTIVE PATCH TOKENIZATION - BMI PREDICTION

✓ Using GPU: Tesla T4
  Memory: 15.83 GB

Configuration:
  Folds:          3
  Epochs:         30
  Batch size:     48
  Learning rate:  3e-05
  Loss function:  huber
  Mixed precision: True
  Output dir:     /kaggle/working/vit_adaptive_checkpoints


TRAINING FOLD 1/3


Fold 1 dataset sizes:
  Train: 39,655 samples
  Val:   19,828 samples

...

Epoch 14:   0%|          | 0/826 [00:00<?, ?it/s]

Validating:   0%|          | 0/207 [00:00<?, ?it/s]

Epoch 14/30 | Time: 377.3s | Train Loss: 0.1649 | Val Loss: 0.1889 | MAE: 0.5903 | RMSE: 0.6146 | R²: 0.9836

  Early stopping triggered at epoch 14
  Best epoch was 7


Validating:   0%|          | 0/207 [00:00<?, ?it/s]


FOLD 3 RESULTS
Training time:  88.24 minutes
MAE:            0.1526
MSE:            0.0875
RMSE:           0.2958
R²:             0.9962


GENERATING PLOTS AND SAVING RESULTS

✓ Training plots saved to: /kaggle/working/logs/training_results.png
✓ Results saved to: /kaggle/working/logs/results.json

FINAL CROSS-VALIDATION RESULTS
Metric     Mean         Std          Min          Max         
----------------------------------------------------------------------
MAE        0.1477       0.0058       0.1396       0.1526      
RMSE       0.2578       0.0274       0.2321       0.2958      
R²         0.9971       0.0006       0.9962       0.9977      

Total training time: 254.90 minutes (4.25 hours)

✓ Training complete!
✓ Models saved in: /kaggle/working/vit_adaptive_checkpoints
✓ Logs saved in: /kaggle/working/logs
